<a href="https://colab.research.google.com/github/tusharsingh3199/3D-Brain-Tumor-Segmentation/blob/main/3D_UNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Import Library, Download Data, Patient Setup, EDA**

In [2]:
import os
import zipfile
import random
import nibabel as nib
from ipywidgets import interact, IntSlider

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf


In [3]:
!kaggle datasets download --d aryashah2k/brain-tumor-segmentation-brats-2019

Dataset URL: https://www.kaggle.com/datasets/aryashah2k/brain-tumor-segmentation-brats-2019
License(s): CC0-1.0
brain-tumor-segmentation-brats-2019.zip: Skipping, found more recently modified local copy (use --force to force download)


In [13]:
Source_File = "/content/brain-tumor-segmentation-brats-2019.zip"
Data_Path = "/content/MICCAI_BraTS_2019_Data_Training"

if not os.path.exists(Data_Path):
    os.makedirs(Data_Path)
    with zipfile.ZipFile(Source_File, 'r') as zip_ref:
        zip_ref.extractall()

def patients_dir(Data_Path):
    patient_dir = []
    for grade in ["HGG", "LGG"]:
        grade_dir = os.path.join(Data_Path, grade)
        patients = sorted(os.listdir(grade_dir))
        patient_paths = [os.path.join(grade_dir, p) for p in patients]

        for path in patient_paths:
            patient_dir.append(path)

    return patient_dir

patients_dir = patients_dir(Data_Path)
random.shuffle(patients_dir)


In [ ]:
def EDA(patients_dir):
    patient_dir = random.choice(patients_dir)
    img = []
    seg = []
    for modality in ["t1", "t1ce", "t2", "flair"]:
        v = nib.load(os.path.join(patient_dir, f"{os.path.basename(patient_dir)}_{modality}.nii")).get_fdata().astype(np.float32)
        v = (v - v.min()) / (v.max() - v.min() + 1e-8)
        img.append(v)

    seg = nib.load(os.path.join(patient_dir, f"{os.path.basename(patient_dir)}_seg.nii")).get_fdata().astype(np.int32)

    def view_slice(z):
        plt.figure(figsize=(12, 6))
        for i in range(4):
            plt.subplot(2, 4, i+1)
            plt.imshow(img[i][:, :, z], cmap="gray")
            plt.axis("off")

            plt.subplot(2, 4, 5+i)
            plt.imshow(img[i][:, :, z], cmap="gray")
            plt.axis("off")

            slice_seg = seg[:, :, z]
            masked_seg = np.ma.masked_where(slice_seg == 0, slice_seg)
            plt.imshow(masked_seg, cmap="jet", alpha=0.7, vmin=0, vmax=4)

        plt.show()

    interact(view_slice, z=IntSlider(min=0, max=154, step=1, value=75))

EDA(patients_dir)

# **Preparing the Dataset for 3D UNet, SWIN Transformer**

In [22]:
train_dir = patients_dir[:int(len(patients_dir)*0.8)]
val_dir = patients_dir[int(len(patients_dir)*0.8):int(len(patients_dir)*0.9)]
test_dir = patients_dir[int(len(patients_dir)*0.9):]

def load_patient(patient_dir):
    patient_dir = patient_dir.numpy().decode()
    images = []
    for modality in ["t1", "t1ce", "t2", "flair"]:
        v = nib.load(os.path.join(patient_dir, f"{os.path.basename(patient_dir)}_{modality}.nii")).get_fdata().astype(np.float32)
        v = (v - v.min()) / (v.max() - v.min() + 1e-8)
        images.append(v)

    image = np.stack(images, axis=-1).astype(np.float32)
    mask = nib.load(os.path.join(patient_dir, f"{os.path.basename(patient_dir)}_seg.nii")).get_fdata().astype(np.int32)
    mask[mask == 4] = 3

    patch_size = (128, 128, 128)
    tumor = np.argwhere(mask > 0)

    if len(tumor) > 0:
        cx, cy, cz = tumor[np.random.randint(len(tumor))]
        x = np.clip(cx - patch_size[0] // 2, 0, image.shape[0] - patch_size[0])
        y = np.clip(cy - patch_size[1] // 2, 0, image.shape[1] - patch_size[1])
        z = np.clip(cz - patch_size[2] // 2, 0, image.shape[2] - patch_size[2])
    else:
        x = np.random.randint(0, image.shape[0] - patch_size[0] + 1)
        y = np.random.randint(0, image.shape[1] - patch_size[1] + 1)
        z = np.random.randint(0, image.shape[2] - patch_size[2] + 1)

    image = image[x:x+patch_size[0], y:y+patch_size[1], z:z+patch_size[2], :]
    mask = mask[x:x+patch_size[0], y:y+patch_size[1], z:z+patch_size[2]]

    return image, mask

def tf_load_patient(patient_dir):
    image, mask = tf.py_function(load_patient, [patient_dir], [tf.float32, tf.int32])
    image.set_shape((128, 128, 128, 4))
    mask.set_shape((128, 128, 128))
    return image, mask


train_dataset = (tf.data.Dataset.from_tensor_slices(train_dir).map(tf_load_patient, num_parallel_calls=tf.data.AUTOTUNE)
                  .batch(1).prefetch(tf.data.AUTOTUNE))

val_dataset = (tf.data.Dataset.from_tensor_slices(val_dir).map(tf_load_patient, num_parallel_calls=tf.data.AUTOTUNE)
                  .batch(1).prefetch(tf.data.AUTOTUNE))

test_dataset = (tf.data.Dataset.from_tensor_slices(test_dir).map(tf_load_patient)
                  .batch(1))

# **3D UNet Model Architecture**

In [29]:
def conv_block(x, filters):
    x = tf.keras.layers.Conv3D(filters, 3, padding="same")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU()(x)

    x = tf.keras.layers.Conv3D(filters, 3, padding="same")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU()(x)

    return x

def decoder_block(x, skip, filters):
    x = tf.keras.layers.Conv3DTranspose(filters, kernel_size=2, strides=2, padding="same")(x)
    x = tf.keras.layers.Concatenate()([x, skip])
    x = conv_block(x, filters)
    return x

def Model_3UNet(input_shape=(128, 128, 128, 4), num_classes=4):
    inputs = tf.keras.Input(input_shape)

    s1 = conv_block(inputs, 16)
    p1 = tf.keras.layers.MaxPool3D(2)(s1)

    s2 = conv_block(p1, 32)
    p2 = tf.keras.layers.MaxPool3D(2)(s2)

    s3 = conv_block(p2, 64)
    p3 = tf.keras.layers.MaxPool3D(2)(s3)

    s4 = conv_block(p3, 128)
    p4 = tf.keras.layers.MaxPool3D(2)(s4)

    b = conv_block(p4, 256)

    d1 = decoder_block(b, s4, 128)
    d2 = decoder_block(d1, s3, 64)
    d3 = decoder_block(d2, s2, 32)
    d4 = decoder_block(d3, s1, 16)

    outputs = tf.keras.layers.Conv3D(num_classes, kernel_size=1, activation="softmax")(d4)
    return tf.keras.Model(inputs, outputs, name="3D_U-Net")


model = Model_3UNet()
model.summary()

Model: "3D_U-Net"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 128, 4)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv3d_38 (Conv3D)  │ (None, 128, 128,  │      1,744 │ input_layer_2[0]… │
│                     │ 128, 16)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 128,  │         64 │ conv3d_38[0][0]   │
│ (BatchNormalizatio… │ 128, 16)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_36 (ReLU)     │ (None, 128, 128,  │          0 │ batch_normalizat… │
│                     │ 128, 16)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv3d_39 (Conv3D)  │ (None, 128, 128,  │      6,928 │ re_lu_36[0][0]    │
│                     │ 128, 16)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 128,  │         64 │ conv3d_39[0][0]   │
│ (BatchNormalizatio… │ 128, 16)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_37 (ReLU)     │ (None, 128, 128,  │          0 │ batch_normalizat… │
│                     │ 128, 16)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling3d_8     │ (None, 64, 64,    │          0 │ re_lu_37[0][0]    │
│ (MaxPooling3D)      │ 64, 16)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv3d_40 (Conv3D)  │ (None, 64, 64,    │     13,856 │ max_pooling3d_8[… │
│                     │ 64, 32)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │        128 │ conv3d_40[0][0]   │
│ (BatchNormalizatio… │ 64, 32)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_38 (ReLU)     │ (None, 64, 64,    │          0 │ batch_normalizat… │
│                     │ 64, 32)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv3d_41 (Conv3D)  │ (None, 64, 64,    │     27,680 │ re_lu_38[0][0]    │
│                     │ 64, 32)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │        128 │ conv3d_41[0][0]   │
│ (BatchNormalizatio… │ 64, 32)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_39 (ReLU)     │ (None, 64, 64,    │          0 │ batch_normalizat… │
│                     │ 64, 32)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling3d_9     │ (None, 32, 32,    │          0 │ re_lu_39[0][0]    │
│ (MaxPooling3D)      │ 32, 32)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv3d_42 (Conv3D)  │ (None, 32, 32,    │     55,360 │ max_pooling3d_9[… │
│                     │ 32, 64)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        256 │ conv3d_42[0][0] 

 Total params: 5,652,148 (21.56 MB)

 Trainable params: 5,649,204 (21.55 MB)

 Non-trainable params: 2,944 (11.50 KB)

# **Model Training, Dice Score, Training Graphs**

In [38]:
ce = tf.keras.losses.SparseCategoricalCrossentropy()

def Dice(y_true, y_pred):

    y_true = tf.one_hot(tf.cast(y_true, tf.int32), depth=4)
    y_pred = tf.one_hot(tf.argmax(y_pred, axis=-1), depth=4)

    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    smooth = 1e-6

    intersection = tf.reduce_sum(y_true * y_pred, axis=[1,2,3])
    union = tf.reduce_sum(y_true + y_pred, axis=[1,2,3])

    return tf.reduce_mean((2. * intersection + smooth) / (union + smooth))


def Loss(y_true, y_pred):

    y_true = tf.one_hot(tf.cast(y_true, tf.int32), depth=4)
    y_true = tf.cast(y_true, tf.float32)

    smooth = 1e-6

    intersection = tf.reduce_sum(y_true * y_pred, axis=[1,2,3])
    union = tf.reduce_sum(y_true + y_pred, axis=[1,2,3])

    dice_loss = 1 - tf.reduce_mean((2. * intersection + smooth) / (union + smooth))

    return dice_loss + ce(y_true, y_pred)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint("3D_UNet.keras", save_best_only=True, monitor="val_Dice", mode="max"),
    tf.keras.callbacks.EarlyStopping(monitor="val_Dice", mode="max", patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_Loss", factor=0.5, patience=3)
]

model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss=Loss, metrics=[Dice])

history = model.fit(train_dataset, validation_data=val_dataset, epochs=10, callbacks=callbacks)

Epoch 1/10
268/268 ━━━━━━━━━━━━━━━━━━━━ 544s 2s/step - dice_score: 0.4502 - loss: 1.2346 - val_dice_score: 0.3334 - val_loss: 1.2123 - learning_rate: 1.0000e-04
Epoch 2/10
268/268 ━━━━━━━━━━━━━━━━━━━━ 392s 1s/step - dice_score: 0.5699 - loss: 0.9690 - val_dice_score: 0.4456 - val_loss: 1.2137 - learning_rate: 1.0000e-04
Epoch 3/10
268/268 ━━━━━━━━━━━━━━━━━━━━ 402s 1s/step - dice_score: 0.6406 - loss: 0.8260 - val_dice_score: 0.4298 - val_loss: 1.3715 - learning_rate: 1.0000e-04
Epoch 4/10
268/268 ━━━━━━━━━━━━━━━━━━━━ 446s 2s/step - dice_score: 0.6708 - loss: 0.7278 - val_dice_score: 0.4428 - val_loss: 1.2584 - learning_rate: 1.0000e-04
Epoch 5/10
268/268 ━━━━━━━━━━━━━━━━━━━━ 445s 2s/step - dice_score: 0.6892 - loss: 0.6565 - val_dice_score: 0.4462 - val_loss: 1.2898 - learning_rate: 1.0000e-04
Epoch 6/10
268/268 ━━━━━━━━━━━━━━━━━━━━ 489s 2s/step - dice_score: 0.7054 - loss: 0.5947 - val_dice_score: 0.4599 - val_loss: 1.2600 - learning_rate: 1.0000e-04
Epoch 7/10
268/268 ━━━━━━━━━━━━━━━

In [ ]:
dice = history.history["Dice"]
val_dice = history.history["val_Dice"]
loss = history.history["Loss"]
val_loss = history.history["val_Loss"]

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.plot(dice, label="Training Dice")
plt.plot(val_dice, label="Validation Dice")
plt.xlabel("Epoch")
plt.ylabel("Dice")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(loss, label="Training Loss")
plt.plot(val_loss, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

model.evaluate(test_dataset)
model.save("3D_UNet.keras")
model = tf.keras.models.load_model("3D_UNet.keras", custom_objects={"Loss": Loss, "Dice": Dice})

# **Model Prediction and Results**

In [ ]:
def MRI_Results(patients_dir, model):
    patient_dir = random.choice(patients_dir)
    images = []
    patient_id = os.path.basename(patient_dir)

    for modality in ["t1", "t1ce", "t2", "flair"]:
        volume = nib.load(os.path.join(patient_dir, f"{patient_id}_{modality}.nii")).get_fdata().astype(np.float32)
        volume = (volume - volume.min()) / (volume.max() - volume.min() + 1e-8)
        images.append(volume)

    image = np.stack(images, axis=-1)
    mask = nib.load(os.path.join(patient_dir, f"{patient_id}_seg.nii")).get_fdata().astype(np.int32)
    mask[mask == 4] = 3

    image = image[61:189, 61:189, 11:139]
    mask = mask[61:189, 61:189, 11:139]

    prediction = model.predict(np.expand_dims(image, axis=0), verbose=0)
    prediction = np.argmax(prediction[0], axis=-1)

    def view_slice(z):
        plt.figure(figsize=(15,9))
        titles = ["T1","T1CE","T2","FLAIR"]

        for i in range(4):

            plt.subplot(3,4,i+1)
            plt.imshow(image[:,:,z,i], cmap="gray")
            plt.title(titles[i])
            plt.axis("off")

            plt.subplot(3,4,5+i)
            plt.imshow(image[:,:,z,i], cmap="gray")
            plt.imshow(np.ma.masked_where(mask[:,:,z]==0, mask[:,:,z]),
                       cmap="jet", alpha=0.6, vmin=0, vmax=3)
            plt.title("Ground Truth")
            plt.axis("off")

            plt.subplot(3,4,9+i)
            plt.imshow(image[:,:,z,i], cmap="gray")
            plt.imshow(np.ma.masked_where(prediction[:,:,z]==0, prediction[:,:,z]),
                       cmap="jet", alpha=0.6, vmin=0, vmax=3)
            plt.title("Prediction")
            plt.axis("off")

        plt.tight_layout()
        plt.show()

    interact(view_slice, z=IntSlider(min=0, max=127, value=64))


MRI_Results(patients_dir, model)